# Example-image figure from the real datasets (about 10 min, no GPU needed)
Builds the figure of example tissue images for the paper directly from the original LC25000 and LungHist700 files:
(a) one LC25000 image per class, (b) four images from a single LC25000 source-tile group (showing why image-level splitting leaks),
(c) one LungHist700 image per class, and (d) one LungHist700 image with the 12-tile grid used for external inference.

**How to run (Google Colab):** a CPU runtime is enough (*Runtime → Change runtime type → CPU*), so it can run while another notebook
uses the GPU. Then *Runtime → Run all* and allow Google Drive access. At the end `example_images_figure.zip` is downloaded
(figure as PDF and 600-dpi PNG/TIFF, plus the original example images). **Send the zip to Claude.**

In [ ]:
# %% Settings
import os
TEST_MODE = os.environ.get("GS_TEST_MODE") == "1"
TILE = 400
GROUPS_COMMIT = "912f342fe2a781e847f395d79982dad4d3d425ae"
GROUPS_SHA256 = "2ff9e992f759f265dfc9aa2363de0dfcc8e6559884d8bb02a4c925214fdf7e23"
GROUPS_URL = f"https://raw.githubusercontent.com/GeorgeBatch/LC25000-clean/{GROUPS_COMMIT}/kaggle/lc25000_image_groups.csv"
LH_URL   = "https://ndownloader.figshare.com/files/45206104"
SAVE_DIR = "/content/drive/MyDrive/example_images_figure"
DATA_DIR = None
LH_ROOT  = "/content/lunghist700"
SEEDS = [42]
if TEST_MODE:
    DATA_DIR, LH_ROOT, SAVE_DIR = os.environ["GS_DATA"], os.environ["GS_LH"], os.environ["GS_OUT"]
    GROUPS_URL = os.environ["GS_GROUPS"]

In [ ]:
# %% Imports
import json, hashlib, glob, re, subprocess, random
import numpy as np, pandas as pd, cv2
CLASSES = ["lung_aca", "lung_n", "lung_scc"]
if not TEST_MODE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# %% LC25000 and source-tile groups (identical to the paper's notebooks)
def find_lung_dir(root):
    for d, ds, _ in os.walk(root):
        if all(c in ds for c in CLASSES): return d
    raise FileNotFoundError(root)
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = find_lung_dir(kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images"))
else:
    DATA_DIR = find_lung_dir(DATA_DIR)
rows = []
for label, c in enumerate(CLASSES):
    for f in sorted(glob.glob(os.path.join(DATA_DIR, c, "*"))):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            with open(f, "rb") as fh:
                rows.append({"path": f, "label": label, "cls": c, "filename": os.path.basename(f), "md5": hashlib.md5(fh.read()).hexdigest()})
df_all = pd.DataFrame(rows)
df = df_all.drop_duplicates("md5", keep="first").reset_index(drop=True)
conflict = df_all.groupby("md5")["label"].nunique()
df = df[~df["md5"].isin(conflict[conflict > 1].index)].reset_index(drop=True)
groups = pd.read_csv(GROUPS_URL)
if not TEST_MODE:
    import urllib.request
    assert hashlib.sha256(urllib.request.urlopen(GROUPS_URL).read()).hexdigest() == GROUPS_SHA256, "grouping file changed - send this message to Claude"
groups = groups[groups["label"].isin(CLASSES)][["filename", "label", "group_id"]].rename(columns={"label": "g_label"})
df = df.merge(groups, on="filename", how="left")
assert df["group_id"].notna().all(), "images without group - send this message to Claude"
assert (df["g_label"] == df["cls"]).all()
df["group_id"] = df["group_id"].astype(int)
print(f"LC25000 lung images kept: {len(df)} | groups: {df['group_id'].nunique()}")



In [ ]:
# %% LungHist700 index (external only; no tuning)
if not glob.glob(LH_ROOT + "/**/*.jp*g", recursive=True):
    os.makedirs(LH_ROOT, exist_ok=True); rar = "/content/LungHist700.rar"
    if not os.path.exists(rar): subprocess.run(["wget", "-q", "-O", rar, LH_URL], check=True)
    if subprocess.run(["which", "unrar"], capture_output=True).returncode != 0:
        subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True)
    if subprocess.run(["unrar", "x", "-o+", "-inul", rar, LH_ROOT + "/"]).returncode != 0:
        subprocess.run(["7z", "x", "-y", f"-o{LH_ROOT}", rar], check=True, stdout=subprocess.DEVNULL)
rows = []
for f in sorted(glob.glob(LH_ROOT + "/**/*.jp*g", recursive=True)):
    m = re.match(r"^(aca|nor|scc)(?:_(bd|md|pd))?_(20x|40x)_(\d+)\.jpe?g$", os.path.basename(f).lower())
    if m:
        sup, sub, res_, iid = m.groups()
        rows.append({"path": f, "superclass": sup, "subclass": sub or "", "resolution": res_, "image_id": int(iid),
                     "label": {"aca": 0, "nor": 1, "scc": 2}[sup]})
ext = pd.DataFrame(rows)
for c in glob.glob(LH_ROOT + "/**/*.csv", recursive=True):
    meta = pd.read_csv(c)
    if {"superclass", "resolution", "image_id", "patient_id"} <= set(meta.columns):
        meta = meta.copy()
        if "subclass" not in meta: meta["subclass"] = ""
        meta["subclass"] = meta["subclass"].fillna("").astype(str).str.lower().replace("nan", "")
        meta["superclass"] = meta["superclass"].astype(str).str.lower(); meta["resolution"] = meta["resolution"].astype(str).str.lower()
        meta["image_id"] = meta["image_id"].astype(int)
        ext = ext.merge(meta[["superclass", "subclass", "resolution", "image_id", "patient_id"]],
                        on=["superclass", "subclass", "resolution", "image_id"], how="left"); break
assert "patient_id" in ext and ext["patient_id"].notna().all(), "patient IDs missing - send this message to Claude"
y_ext, pat = ext["label"].values, ext["patient_id"].values
T = 12
print(f"LungHist700: {len(ext)} images, {ext['patient_id'].nunique()} patients")

def ext_tiles(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    out = [img[yy:yy + TILE, xx:xx + TILE] for yy in range(0, H - TILE + 1, TILE) for xx in range(0, W - TILE + 1, TILE)]
    assert len(out) == T, f"{path}: {len(out)} tiles instead of 12 - send this message to Claude"
    return out



In [ ]:
# %% Build the figure
import matplotlib
if TEST_MODE: matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
plt.rcParams.update({"font.family": "serif", "font.size": 8})
rng = random.Random(7)
rgb = lambda p: cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
NAMES = {"lung_aca": "Adenocarcinoma", "lung_n": "Benign lung", "lung_scc": "Squamous cell carcinoma"}
LHN = {"aca": "Adenocarcinoma", "nor": "Normal lung", "scc": "Squamous cell carcinoma"}

# (a) one LC25000 image per class (fixed random choice)
pick_a = {c: df[df["cls"] == c].sort_values("filename").iloc[rng.randrange((df["cls"] == c).sum())] for c in CLASSES}
# (b) four images of one adenocarcinoma source-tile group with at least four images
sizes = df[df["cls"] == "lung_aca"].groupby("group_id").size()
gid = sorted(sizes[sizes >= 4].index)[rng.randrange((sizes >= 4).sum())]
grp = df[df["group_id"] == gid].sort_values("filename").head(4)
# (c) one LungHist700 image per class at 20x (fixed random choice), (d) one 40x image with the tile grid
pick_c = {s: ext[(ext["superclass"] == s) & (ext["resolution"] == "20x")].sort_values("path").iloc[0] for s in ["aca", "nor", "scc"]}
cand = ext[ext["resolution"] == "40x"].sort_values("path"); pick_d = cand.iloc[rng.randrange(len(cand))] if len(cand) else ext.iloc[0]

fig = plt.figure(figsize=(7.2, 7.6))
gs = fig.add_gridspec(3, 4, height_ratios=[1, 1, 1.05], hspace=0.32, wspace=0.06)
def show(ax, img, title):
    ax.imshow(img); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=7.5, pad=3)
    for s in ax.spines.values(): s.set_linewidth(0.4)
for j, c in enumerate(CLASSES):
    show(fig.add_subplot(gs[0, j]), rgb(pick_a[c]["path"]), NAMES[c])
fig.text(0.075, 0.905, "(a) LC25000 lung subset (768 × 768 pixels)", fontsize=8.5, weight="bold")
ax = fig.add_subplot(gs[0, 3]); ax.axis("off")
ax.text(0.05, 0.5, "One image per class.\nLC25000 contains\n250 source images per\nclass, each expanded\nby augmentation.", fontsize=7, va="center")
for j, (_, r) in enumerate(grp.iterrows()):
    show(fig.add_subplot(gs[1, j]), rgb(r["path"]), r["filename"])
fig.text(0.075, 0.625, f"(b) Four LC25000 images from one source-tile group (adenocarcinoma, group {gid})", fontsize=8.5, weight="bold")
for j, s in enumerate(["aca", "nor", "scc"]):
    r = pick_c[s]; show(fig.add_subplot(gs[2, j]), rgb(r["path"]), LHN[s])
ax = fig.add_subplot(gs[2, 3]); img = rgb(pick_d["path"]); show(ax, img, f"{LHN[pick_d['superclass']]}, {pick_d['resolution']}")
H, W = img.shape[:2]
for yy in range(0, H - TILE + 1, TILE):
    for xx in range(0, W - TILE + 1, TILE):
        ax.add_patch(Rectangle((xx, yy), TILE, TILE, fill=False, ec="white", lw=0.8))
fig.text(0.075, 0.335, "(c) LungHist700 external cohort, 20× (1200 × 1600 pixels)          (d) 12-tile grid", fontsize=8.5, weight="bold")
for ext_ in ["pdf", "png"]:
    fig.savefig(os.path.join(SAVE_DIR, f"fig_examples.{ext_}"), dpi=600, bbox_inches="tight")
from PIL import Image
Image.open(os.path.join(SAVE_DIR, "fig_examples.png")).convert("RGB").save(os.path.join(SAVE_DIR, "fig_examples.tif"), compression="tiff_lzw", dpi=(600, 600))
plt.show() if not TEST_MODE else None; plt.close(fig)

# keep the original example files and a record of what was chosen
import shutil
od = os.path.join(SAVE_DIR, "originals"); os.makedirs(od, exist_ok=True)
for p in [pick_a[c]["path"] for c in CLASSES] + list(grp["path"]) + [pick_c[s]["path"] for s in pick_c] + [pick_d["path"]]:
    shutil.copy(p, od)
json.dump({"lc25000_class_examples": {c: pick_a[c]["filename"] for c in CLASSES}, "source_group": int(gid),
           "source_group_files": list(grp["filename"]), "group_size": int(sizes[gid]),
           "lunghist700_examples": {s: os.path.basename(pick_c[s]["path"]) for s in pick_c},
           "lunghist700_tiling_example": os.path.basename(pick_d["path"])},
          open(os.path.join(SAVE_DIR, "examples.json"), "w"), indent=1)
print("Saved:", sorted(os.listdir(SAVE_DIR)))
if not TEST_MODE:
    subprocess.run(f"cd {os.path.dirname(SAVE_DIR)} && zip -qr /content/example_images_figure.zip {os.path.basename(SAVE_DIR)}", shell=True)
    from google.colab import files; files.download("/content/example_images_figure.zip")